In [2]:
import sys
sys.path.append("..")

import pandas as pd
import plotly.express as px
from src.data_pipeline.stock_data import load_prices

prices = load_prices()
prices["date"] = pd.to_datetime(prices["date"])
print(prices.shape)

(12540, 7)


In [3]:
close = prices.pivot(index="date", columns="ticker", values="close")
returns = close.pct_change().dropna()
close.tail(3)

ticker,AAPL,AMZN,GOOGL,JNJ,JPM,META,MSFT,NVDA,TSLA,XOM
date,,,,,,,,,,
2026-07-16,333.260010,249.889999,354.459991,249.970001,343.149994,664.539978,401.100006,207.399994,391.059998,145.949997
2026-07-17,333.739990,247.229996,346.769989,253.039993,341.100006,646.010010,393.820007,202.809998,380.839996,147.360001
2026-07-20,326.589996,249.990005,351.989990,248.820007,338.869995,645.849976,402.290009,203.279999,369.570007,148.360001


In [4]:
norm = close / close.iloc[0] * 100
fig = px.line(norm, title="₹100 invested 5 years ago — who won?")
fig.show()

In [5]:
fig = px.histogram(returns["TSLA"], nbins=100, title="TSLA daily returns")
fig.show()
print(returns.std().round(4).sort_values().rename("daily volatility"))
print(returns.kurtosis().round(1).sort_values().rename("kurtosis (normal≈0)"))

ticker
JNJ      0.0109
JPM      0.0154
XOM      0.0168
MSFT     0.0171
AAPL     0.0175
GOOGL    0.0200
AMZN     0.0225
META     0.0281
NVDA     0.0326
TSLA     0.0374
Name: daily volatility, dtype: float64
ticker
XOM       1.4
TSLA      3.0
GOOGL     3.2
MSFT      3.5
JNJ       4.4
NVDA      4.7
AMZN      4.8
JPM       5.1
AAPL      6.7
META     19.0
Name: kurtosis (normal≈0), dtype: float64


In [6]:
vol = returns.rolling(20).std()
fig = px.line(vol[["TSLA", "JNJ", "NVDA"]], title="20-day rolling volatility")
fig.show()

In [7]:
fig = px.imshow(returns.corr().round(2), text_auto=True, color_continuous_scale="RdBu_r",
                title="Daily return correlations, 5 years")
fig.show()

In [10]:
for t in ["TSLA", "NVDA", "META"]:
    best, worst = returns[t].idxmax(), returns[t].idxmin()
    print(f"{t}: best {best.date()} ({returns[t].max():+.1%}) | worst {worst.date()} ({returns[t].min():+.1%})")

TSLA: best 2025-04-09 (+22.7%) | worst 2025-03-10 (-15.4%)
NVDA: best 2023-05-25 (+24.4%) | worst 2025-01-27 (-17.0%)
META: best 2023-02-02 (+23.3%) | worst 2022-02-03 (-26.4%)


## 📊 Findings — Price EDA (Week 1 close)
*(Written by mentor during illness accommodation, Jul 22 — computed from our actual data. Review when recovered.)*

**1. The 5-year race has one runaway winner.** ₹100 invested 5 years ago: **NVDA → ₹1,051** (10.5×) — nobody else is close. The surprise runner-up is **XOM → ₹309** (the 2021-22 energy rally), beating GOOGL (278), AAPL (230). Bottom: AMZN (139) and MSFT (149). Lesson: mega-cap ≠ mega-return; the winners of the *last* 5 years weren't obvious 5 years ago.

**2. Every stock has a volatility personality, and it's stable.** Daily volatility runs from **TSLA 3.74%** and **NVDA 3.26%** down to **JNJ 1.09%** — a 3.4× gap. In the rolling-vol chart TSLA's line lives permanently above JNJ's, and all lines spike *together* in stress periods (volatility clustering). These personalities feed Day 13's risk score.

**3. Fat tails are real — the bell curve is a lie.** Kurtosis (normal distribution ≈ 0): **META 19.0**, AAPL 6.7, JPM 5.1 … lowest is XOM at 1.4 — *still above normal*. Translation: extreme days happen far more often than a bell curve predicts. META's 19 comes largely from **2022-02-03: −26.4% in one day** (first-ever user decline + weak guidance — at the time the largest one-day value loss in US market history). Any Week 2 model that assumes normal returns will be blindsided.

**4. Correlation shows two different worlds.** Tech↔tech average correlation: **0.55** (they move as a herd). But **JNJ vs tech: 0.01** and **XOM vs tech: 0.08** — essentially independent. JPM sits between (0.33). This is what diversification literally is, and why our 10-ticker universe (not 10 tech names) was the right design.

**5. Every extreme day has a story — this is the chatbot's future job.**
- META **−26.4%** (2022-02-03): users declined for the first time; guidance miss
- NVDA **+24.4%** (2023-05-25): the AI-boom earnings blowout — data-center guidance shattered estimates
- NVDA **−17.0%** (2025-01-27): the DeepSeek shock — cheap-AI fears hit the AI-hardware king
- TSLA **+22.7%** (2025-04-09): the tariff-pause relief rally
Connecting price moves to news manually here is exactly what FinSentinel automates in Week 3.

**Implications for Week 2:** model *returns*, never prices (scale + stationarity); expect fat tails; volatility and correlation structure are features, not noise; a "always predict up" baseline will be hard to beat — respect it.
